# Extração Scopus (pybliometrics) — Produção Bibliográfica

Notebook original (`Quick-Start.ipynb`) era a demonstração padrão da biblioteca
[`pyscopus`](http://zhiyzuo.github.io/python-scopus/). Essa lib está
desatualizada e vinha causando problemas, então esta versão migra para
[`pybliometrics`](https://github.com/pybliometrics-dev/pybliometrics) — wrapper
oficial e ativamente mantido para as APIs Scopus/ScienceDirect/SciVal da Elsevier.

O pipeline continua o mesmo: dada uma lista de pessoas (com `id_lattes` e
`scopus_author_id`), para cada uma buscamos todas as publicações indexadas na
Scopus e produzimos DataFrames **no mesmo formato de saída do `analyse.ipynb`**:

- `df_artigos_periodico_scopus` → mesmo schema de `df_artigos_final` (artigos de periódico)
- `df_artigos_congresso_scopus` → mesmo schema de `df_artigos_congresso_final` (trabalhos de congresso/proceedings)

Diferente da versão com `pyscopus`, a classe `ScopusSearch` do `pybliometrics`
já retorna, em uma única chamada por pessoa, `doi`, `author_names`, `issn`,
`pageRange` e `aggregationType` — não é mais necessário um segundo request por
publicação só para complementar DOI/ISSN.

## 1. Instalação e Configuração

In [1]:
%pip install pybliometrics python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import pybliometrics
from pybliometrics.scopus import ScopusSearch
#from pybliometrics.scopus.exception import Scopus401Error, ScopusQueryError

load_dotenv()

# Sua API Key da Elsevier (https://dev.elsevier.com/) — registre uma chave e
# guarde em uma variável de ambiente SCOPUS_API_KEY (ex.: em um arquivo .env).
# Na primeira execução, se nenhuma chave for encontrada, o pybliometrics.init()
# pede a chave interativamente e cria o arquivo de configuração em
# ~/.config/pybliometrics.cfg (não é necessário repetir isso depois).
API_KEY = os.getenv('SCOPUS_API_KEY')
INST_TOKEN = os.getenv('SCOPUS_INST_TOKEN')  # opcional, só se sua instituição usar token

if API_KEY:
    pybliometrics.init(keys=[API_KEY], inst_tokens=[INST_TOKEN] if INST_TOKEN else None)
else:
    # Lê o arquivo de configuração já existente (~/.config/pybliometrics.cfg)
    # ou pede a chave de forma interativa caso ainda não exista
    pybliometrics.init()

print("Cliente Scopus (pybliometrics) inicializado com sucesso!")

Cliente Scopus (pybliometrics) inicializado com sucesso!


## 2. Lista de pessoas a extrair

Cada pessoa precisa ter:

- `id_lattes`: a chave que une todas as fontes (Lattes, ORCID, Scopus) — é o que vai virar a FK no banco.
- `scopus_author_id`: o Author ID da Scopus da pessoa (ex.: `'57189222659'`), usado só para consultar a API.

Substitua a lista de exemplo abaixo pela sua lista real.

In [5]:
# Exemplo de lista de pessoas. Troque por:
#   df_pessoas_lista = pd.read_csv('lista_pessoas.csv')  # colunas: id_lattes, orcid_id, scopus_author_id
lista_pessoas_scopus = [
    {'id_lattes': '0000000000000001', 'scopus_author_id': '15753781000'},
    # {'id_lattes': '...', 'scopus_author_id': '...'},
]

print(f"Total de pessoas a processar: {len(lista_pessoas_scopus)}")

Total de pessoas a processar: 1


## 3. Extração das publicações de cada autor

Para cada pessoa usamos `ScopusSearch('AU-ID(<scopus_author_id>)')`, que devolve
em `.results` uma lista de namedtuples — uma por publicação — já contendo
`title`, `publicationName`, `coverDate`, `doi`, `issn`, `pageRange`,
`author_names`, `aggregationType`, entre outros.

Classificamos cada publicação em **periódico** ou **congresso/proceedings**
usando `aggregationType`:

- `'Journal'` (e afins: `'Book Series'`) → artigo de periódico
- `'Conference Proceeding'` → trabalho de congresso

> **Nota sobre limites/cache da API:** o `pybliometrics` já armazena em cache
> local os resultados de cada `ScopusSearch`, então execuções repetidas não
> reconsultam a API (use `refresh=True` se quiser forçar atualização). Ainda
> assim, mantemos um pequeno `time.sleep` entre pessoas para respeitar o
> rate-limit da sua chave.

In [6]:
def extrair_ano(cover_date):
    """Extrai o ano (int) de uma string de data tipo '2018-08-01'."""
    if not cover_date or pd.isna(cover_date):
        return pd.NA
    try:
        return int(str(cover_date)[:4])
    except (ValueError, TypeError):
        return pd.NA


lista_artigos_periodico_scopus = []
lista_artigos_congresso_scopus = []

TIPOS_PERIODICO_SCOPUS = {'journal'}
TIPOS_CONGRESSO_SCOPUS = {'conference proceeding'}

for pessoa in lista_pessoas_scopus:
    id_lattes = pessoa['id_lattes']
    scopus_author_id = pessoa['scopus_author_id']

    print(f"Processando Scopus Author ID {scopus_author_id} (id_lattes={id_lattes})...")

    try:
        s = ScopusSearch(f'AU-ID({scopus_author_id})')
    except () as exc:
        print(f"  -> Falha ao buscar publicações de {scopus_author_id}: {exc}")
        continue
    except Exception as exc:
        print(f"  -> Erro inesperado em {scopus_author_id}: {exc}")
        continue

    
    
    publicacoes = s.results or []
    print("OI", publicacoes)
    if not publicacoes:
        print("  -> Nenhuma publicação encontrada.")
        continue

    for pub in publicacoes:
        titulo = pub.title or pd.NA
        revista = pub.publicationName or pd.NA
        ano = extrair_ano(pub.coverDate)
        doi = pub.doi or pd.NA
        issn = pub.issn or pd.NA
        autores = pub.author_names or pd.NA
        tipo_agregacao = (pub.aggregationType or '').strip().lower()

        if tipo_agregacao in TIPOS_CONGRESSO_SCOPUS:
            # --- Trabalho de Congresso / Proceedings ---
            lista_artigos_congresso_scopus.append({
                'id_lattes': id_lattes,
                'titulo_artigo': titulo,
                'ano': ano,
                'doi': doi,
                'autores': autores,
                'titulo_evento_lattes': revista,
                'paginas': pub.pageRange or pd.NA,
                'sigla_evento_google': pd.NA,
                'titulo_evento_google': pd.NA,
                'estrato': pd.NA,
                'tipo_match': 'SCOPUS',
                'coautoria_aluno': pd.NA,
            })
        else:
            # --- Artigo de Periódico (default para os demais aggregationType) ---
            lista_artigos_periodico_scopus.append({
                'id_lattes': id_lattes,
                'titulo_artigo': titulo,
                'titulo_revista_lattes': pd.NA,  # não há contraparte do Lattes aqui
                'ano_pub': ano,
                'doi': doi,
                'autores': autores,
                'match_adequado': pd.NA,
                'coautoria_aluno': pd.NA,
                'id_scopus': pub.eid or pd.NA,
                'titulo_revista_scopus': revista,
                'maior_percentil': pd.NA,
                'codigo_area_maior_percentil': pd.NA,
                'area_maior_percentil': pd.NA,
                'issn': issn,
                'computation_area': pd.NA,
            })

    time.sleep(0.2)

print("\nExtração concluída.")
print(f"Artigos de periódico extraídos da Scopus: {len(lista_artigos_periodico_scopus)}")
print(f"Trabalhos de congresso extraídos da Scopus: {len(lista_artigos_congresso_scopus)}")

Processando Scopus Author ID 15753781000 (id_lattes=0000000000000001)...
OI [Document(eid='2-s2.0-105004445604', doi='10.1111/joes.12702', pii=None, pubmed_id=None, title='Portfolio Optimization for Pension Purposes: Literature Review', subtype='ar', subtypeDescription='Article', creator='Moreira L.', afid='60000036;60042765', affilname='Universidade Federal do Rio de Janeiro;Centro Federal De Educacão Tecnológica Celso Suckow Da Fonseca', affiliation_city='Rio de Janeiro;Rio de Janeiro', affiliation_country='Brazil;Brazil', author_count='3', author_names='Moreira, Leonardo;Santos, Igor Leão dos;Gonzalez, Pedro Henrique', author_ids='59781546200;59781896400;15753781000', author_afids='60042765;60042765;60000036', coverDate='2026-02-01', coverDisplayDate='February 2026', publicationName='Journal of Economic Surveys', issn='09500804', source_id='24353', eIssn='14676419', aggregationType='Journal', volume='40', issueIdentifier='1', article_number=None, pageRange='45-72', description='This

## 4. Consolidação nos DataFrames finais (mesmo schema do `analyse.ipynb`)

As colunas abaixo replicam exatamente:

- `df_artigos_final` → `id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, doi, autores, match_adequado, coautoria_aluno, id_scopus, titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil, area_maior_percentil, issn, computation_area`
- `df_artigos_congresso_final` → `id_lattes, titulo_artigo, ano, doi, autores, titulo_evento_lattes, paginas, sigla_evento_google, titulo_evento_google, estrato, tipo_match, coautoria_aluno`

Os campos que a Scopus não fornece nesta extração (`titulo_revista_lattes`,
`maior_percentil`, `area_maior_percentil`, `computation_area`, `match_adequado`,
`estrato`, `sigla_evento_google`/`titulo_evento_google`) ficam nulos — o
cruzamento de percentil/área e de eventos já é feito pelo próprio
`analyse.ipynb`; se quiser aplicar a mesma lógica aqui, basta reaproveitá-la
passando estes DataFrames no lugar de `df_bib_artigos`/`df_bib_trab_congresso`.

In [5]:
colunas_periodico = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area',
]

colunas_congresso = [
    'id_lattes', 'titulo_artigo', 'ano', 'doi', 'autores',
    'titulo_evento_lattes', 'paginas', 'sigla_evento_google',
    'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno',
]

df_artigos_periodico_scopus = pd.DataFrame(lista_artigos_periodico_scopus, columns=colunas_periodico)
df_artigos_congresso_scopus = pd.DataFrame(lista_artigos_congresso_scopus, columns=colunas_congresso)

# Mesma tipagem usada no analyse.ipynb para permitir o concat sem surpresas
if not df_artigos_periodico_scopus.empty:
    df_artigos_periodico_scopus['ano_pub'] = pd.to_numeric(df_artigos_periodico_scopus['ano_pub'], errors='coerce').astype('Int64')
    df_artigos_periodico_scopus['id_lattes'] = df_artigos_periodico_scopus['id_lattes'].astype(str)
    df_artigos_periodico_scopus['titulo_revista_scopus'] = (
        df_artigos_periodico_scopus['titulo_revista_scopus'].astype(str).str.upper().str.strip()
    )

if not df_artigos_congresso_scopus.empty:
    df_artigos_congresso_scopus['ano'] = pd.to_numeric(df_artigos_congresso_scopus['ano'], errors='coerce').astype('Int64')
    df_artigos_congresso_scopus['id_lattes'] = df_artigos_congresso_scopus['id_lattes'].astype(str)

print("=== df_artigos_periodico_scopus ===")
display(df_artigos_periodico_scopus.head())
df_artigos_periodico_scopus.info()

print("\n=== df_artigos_congresso_scopus ===")
display(df_artigos_congresso_scopus.head())
df_artigos_congresso_scopus.info()

=== df_artigos_periodico_scopus ===


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area
0,0000000000000001,How not to get your paper rejected — From the ...,<NA>,2026,10.1016/j.infsof.2026.108197,"Staron, Miroslaw;Travassos, Guilherme Horta;Ru...",<NA>,<NA>,2-s2.0-105039167479,INFORMATION AND SOFTWARE TECHNOLOGY,<NA>,<NA>,<NA>,09505849,<NA>
1,0000000000000001,Who “controls” where work shall be done? State...,<NA>,2026,10.1016/j.jss.2026.112848,"Smite, Darja;Moe, Nils Brede;Baldassarre, Mari...",<NA>,<NA>,2-s2.0-105035786010,JOURNAL OF SYSTEMS AND SOFTWARE,<NA>,<NA>,<NA>,01641212,<NA>
2,0000000000000001,"Design, execution, and contextual factors shap...",<NA>,2026,10.1016/j.infsof.2026.108136,"Ribeiro, Talita Vieira;de França, Breno Bernar...",<NA>,<NA>,2-s2.0-105034739608,INFORMATION AND SOFTWARE TECHNOLOGY,<NA>,<NA>,<NA>,09505849,<NA>
3,0000000000000001,Preface for “Selected Papers from the 27th Ibe...,<NA>,2026,10.1016/j.scico.2025.103388,"Oliveira, Edson;de Guzmán, Ignacio García Rodr...",<NA>,<NA>,2-s2.0-105015954439,SCIENCE OF COMPUTER PROGRAMMING,<NA>,<NA>,<NA>,01676423,<NA>
4,0000000000000001,Experimental Evaluation of a Checklist-Based I...,<NA>,2025,10.1007/s10664-025-10681-7,"Cerqueira, Diego André;de Mello, Rafael Maiani...",<NA>,<NA>,2-s2.0-105007445503,EMPIRICAL SOFTWARE ENGINEERING,<NA>,<NA>,<NA>,13823256,<NA>


<class 'pandas.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id_lattes                    75 non-null     str   
 1   titulo_artigo                75 non-null     str   
 2   titulo_revista_lattes        0 non-null      object
 3   ano_pub                      75 non-null     Int64 
 4   doi                          69 non-null     str   
 5   autores                      75 non-null     str   
 6   match_adequado               0 non-null      object
 7   coautoria_aluno              0 non-null      object
 8   id_scopus                    75 non-null     str   
 9   titulo_revista_scopus        75 non-null     str   
 10  maior_percentil              0 non-null      object
 11  codigo_area_maior_percentil  0 non-null      object
 12  area_maior_percentil         0 non-null      object
 13  issn                         68 non-null     str

,id_lattes,titulo_artigo,ano,doi,autores,titulo_evento_lattes,paginas,sigla_evento_google,titulo_evento_google,estrato,tipo_match,coautoria_aluno
0,0000000000000001,Aggregating Empirical Evidence from Data Strat...,2025,10.1109/ESEM64174.2025.00049,"Del Rey, Santiago;Dos Santos, Paulo Sergio Med...",International Symposium on Empirical Software ...,12-22,<NA>,<NA>,<NA>,SCOPUS,<NA>
1,0000000000000001,Initial results of a rapid review of the techn...,2024,NaN,"de Paiva, Bruno D.;Esteves, Alexandre C.;de Me...",27th Ibero American Conference on Software Eng...,372-379,<NA>,<NA>,<NA>,SCOPUS,<NA>
2,0000000000000001,Use of Technology Probe in software systems en...,2024,NaN,"Nascimento, Luciana;Galeno, Larissa;Pessoa, Cl...",27th Ibero American Conference on Software Eng...,211-225,<NA>,<NA>,<NA>,SCOPUS,<NA>
3,0000000000000001,A Literature Study on Application Domains and ...,2024,NaN,"da Silva, Fernando N.R.;de Souza, Bruno P.;Tra...",27th Ibero American Conference on Software Eng...,181-195,<NA>,<NA>,<NA>,SCOPUS,<NA>
4,0000000000000001,Towards the Management of the Location and Use...,2023,10.1145/3596454.3597182,"Maia, Vitor Carneiro;De Oliveira, Kathia Marça...",Eics 2023 Companion Companion of the 2023 ACM ...,45-52,<NA>,<NA>,<NA>,SCOPUS,<NA>


<class 'pandas.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_lattes             128 non-null    str   
 1   titulo_artigo         128 non-null    str   
 2   ano                   128 non-null    Int64 
 3   doi                   82 non-null     str   
 4   autores               128 non-null    str   
 5   titulo_evento_lattes  128 non-null    str   
 6   paginas               96 non-null     str   
 7   sigla_evento_google   0 non-null      object
 8   titulo_evento_google  0 non-null      object
 9   estrato               0 non-null      object
 10  tipo_match            128 non-null    str   
 11  coautoria_aluno       0 non-null      object
dtypes: Int64(1), object(4), str(7)
memory usage: 47.7+ KB


## 5. Próximo passo: unificar com `analyse.ipynb` e `orcid.ipynb`

```python
df_periodicos_unificado = pd.concat(
    [df_artigos_final, df_artigos_periodico_orcid, df_artigos_periodico_scopus],
    ignore_index=True,
)

df_congressos_unificado = pd.concat(
    [df_artigos_congresso_final, df_artigos_congresso_orcid, df_artigos_congresso_scopus],
    ignore_index=True,
)
```

Esses DataFrames já estão no formato esperado por `tb_artigo_periodico` e
`tb_artigo_conferencia` no DuckDB (mesmo `INSERT INTO ... (...)` que o
`analyse.ipynb` já define), prontos para subir ao banco. Antes de inserir,
garanta que toda pessoa referenciada em `id_lattes` já exista em
`tb_professores` (a FK exige isso).

Opcionalmente, é possível remover duplicatas entre as fontes (ex.: o mesmo
artigo aparecendo no Lattes e na Scopus) usando `doi` como chave — quando o
`doi` é nulo, cair de volta para `(titulo_artigo, id_lattes)`:

```python
df_periodicos_unificado['chave_dedup'] = df_periodicos_unificado['doi'].fillna(
    df_periodicos_unificado['titulo_artigo'].str.upper().str.strip() + '|' + df_periodicos_unificado['id_lattes']
)
df_periodicos_unificado = df_periodicos_unificado.drop_duplicates(subset=['chave_dedup']).drop(columns=['chave_dedup'])
```